[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 自动微分与反向传播

## 雅可比矩阵

设 ${\bf f}:\mathbb{R}^n\to \mathbb{R}^m$，我们把它（对自变量）的雅可比矩阵定义为：
\begin{align*}
\newcommand{\bbx}{{\bf x}}
\newcommand{\bbv}{{\bf v}}
\newcommand{\bbw}{{\bf w}}
\newcommand{\bbu}{{\bf u}}
\newcommand{\bbf}{{\bf f}}
\newcommand{\bbg}{{\bf g}}
\frac{\partial \bbf}{\partial \bbx} = J_{\bbf}(\bbx) &= \left( \begin{array}{ccc}
\frac{\partial f_1}{\partial x_1}&\dots& \frac{\partial f_1}{\partial x_n}\\
\vdots&&\vdots\\
\frac{\partial f_m}{\partial x_1}&\dots& \frac{\partial f_m}{\partial x_n}
\end{array}\right)\\
&=\left( \frac{\partial \bbf}{\partial x_1},\dots, \frac{\partial \bbf}{\partial x_n}\right)\\
&=\left(
\begin{array}{c}
\nabla f_1(\bbx)^T\\
\vdots\\
\nabla f_m(x)^T
\end{array}\right)
\end{align*}

因此雅可比矩阵 $J_{\bbf}(\bbx)\in \mathbb{R}^{m\times n}$ 是一个从 $\mathbb{R}^n$ 到 $\mathbb{R}^m$ 的线性映射，满足：对 $\bbx,\bbv \in \mathbb{R}^n$ 和 $h\in \mathbb{R}$：
\begin{align*}
\bbf(\bbx+h\bbv) = \bbf(\bbx) + hJ_{\bbf}(\bbx)\bbv +o(h).
\end{align*}
项 $J_{\bbf}(\bbx)\bbv\in \mathbb{R}^m$ 叫做雅可比向量积（**JVP**），它对应如下的解释：把雅可比矩阵看作线性映射 $J_{\bbf}(\bbx):\mathbb{R}^n \to \mathbb{R}^m$，其中 $J_{\bbf}(\bbx)(\bbv)=J_{\bbf}(\bbx)\bbv$。

## 链式复合

在机器学习中，我们计算损失函数对参数的梯度。特别是当参数是高维的时候，损失是一个实数。因此，考虑一个实值函数 $\bbf:\mathbb{R}^n\stackrel{\bbg_1}{\to}\mathbb{R}^m \stackrel{\bbg_2}{\to}\mathbb{R}^d\stackrel{h}{\to}\mathbb{R}$，于是 $\bbf(\bbx) = h(\bbg_2(\bbg_1(\bbx)))\in \mathbb{R}$。我们有
\begin{align*}
\underbrace{\nabla\bbf(\bbx)}_{n\times 1}=\underbrace{J_{\bbg_1}(\bbx)^T}_{n\times m}\underbrace{J_{\bbg_2}(\bbg_1(\bbx))^T}_{m\times d}\underbrace{\nabla h(\bbg_2(\bbg_1(\bbx)))}_{d\times 1}.
\end{align*}
做这个计算时，如果从右边开始：先用一个矩阵乘向量得到一个向量（大小为 $m$），再做一个矩阵乘向量，总共 $O(nm+md)$ 次运算。如果从左边开始做矩阵乘矩阵，则是 $O(nmd+nd)$ 次运算。可以看到，只要 $m\approx d$，从右边开始就要高效得多。不过要注意，从右往左计算需要把 $\bbg_1(\bbx)\in\mathbb{R}^m$ 和 $\bbx\in \mathbb{R}^n$ 的值保存在内存里。

**反向传播**是一个高效的算法，它"从右往左"（也就是向后）计算梯度。具体来说，我们需要计算形如 $J_{\bbf}(\bbx)^T\bbu \in \mathbb{R}^n$（$\bbu \in\mathbb{R}^m$）的量，它可以改写成 $\bbu^T J_{\bbf}(\bbx)$，这就是向量雅可比积（**VJP**），它对应如下的解释：把雅可比矩阵看作线性映射 $J_{\bbf}(\bbx):\mathbb{R}^n \to \mathbb{R}^m$，再与线性映射 $\bbu:\mathbb{R}^m\to \mathbb{R}$ 复合，于是 $\bbu^TJ_{\bbf}(\bbx) = \bbu \circ J_{\bbf}(\bbx)$。

**例子：** 设 $\bbf(\bbx, W) = \bbx W\in \mathbb{R}^b$，其中 $W\in \mathbb{R}^{a\times b}$，$\bbx\in \mathbb{R}^a$。显然有
$$
J_{\bbf}(\bbx) = W^T.
$$
注意，这里我们稍微滥用了一点记号，考虑的是偏函数 $\bbx\mapsto \bbf(\bbx, W)$。为了看清楚，我们可以写 $f_j = \sum_{i}x_iW_{ij}$，于是
$$
\frac{\partial \bbf}{\partial x_i}= \left( W_{i1}\dots W_{ib}\right)^T
$$
再由定义可知
$$
J_{\bbf}(\bbx) = \left( \frac{\partial \bbf}{\partial x_1},\dots, \frac{\partial \bbf}{\partial x_n}\right)=W^T.
$$
现在显然有
$$
J_{\bbf}(W) = \bbx \text{ 因为， } \bbf(\bbx,W+\Delta W) = \bbf(\bbx,W) + \bbx \Delta W.
$$
注意，把 $\bbx$ 放在右边相乘在使用广播时其实很方便：也就是说，不改动上面的数学，我们就能直接处理形状为 $\text{bs}\times a$ 的一批输入向量。

## 实现

在 PyTorch 里，`torch.autograd` 提供了实现任意标量值函数自动微分的类和函数。要创建自定义的 [autograd.Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function)，就继承这个类并实现 `forward()` 和 `backward()` 两个静态方法。下面是一个例子：
```python=
class Exp(Function):
    @staticmethod
    def forward(ctx, i):
        result = i.exp()
        ctx.save_for_backward(result)
        return result
    @staticmethod
    def backward(ctx, grad_output):
        result, = ctx.saved_tensors
        return grad_output * result
# 通过调用 apply 方法来使用它：
output = Exp.apply(input)
```
你可以看看[模块 2b](https://dataflowr.github.io/website/modules/2b-automatic-differentiation) 了解更多这种思路，也可以看[从零实现 MLP](https://dataflowr.github.io/website/homework/1-mlp-from-scratch/)。

### 用函数式的方式做反向传播

这里我们用 `numpy` 实现一种不同的方法，模仿 [JAX](https://jax.readthedocs.io/en/latest/index.html) 的函数式思路，参见 [The Autodiff Cookbook](https://jax.readthedocs.io/en/latest/notebooks/autodiff_cookbook.html#)。

每个函数接受 2 个参数：一个是输入 `x`，另一个是参数 `w`。对每个函数，我们构建 2 个 **vjp** 函数，它们接受一个梯度 $\bbu$ 作为参数，分别对应 $J_{\bbf}(\bbx)$ 和 $J_{\bbf}(\bbw)$，并返回 $J_{\bbf}(\bbx)^T \bbu$ 和 $J_{\bbf}(\bbw)^T \bbu$。总结一下，对 $\bbx \in \mathbb{R}^n$、$\bbw \in \mathbb{R}^d$ 以及 $\bbf(\bbx,\bbw) \in \mathbb{R}^m$，
\begin{align*}
{\bf jvp}_\bbx(\bbu) &= J_{\bbf}(\bbx)^T \bbu, \text{ 其中 } J_{\bbf}(\bbx)\in\mathbb{R}^{m\times n}, \bbu\in \mathbb{R}^m\\
{\bf jvp}_\bbw(\bbu) &= J_{\bbf}(\bbw)^T \bbu, \text{ 其中 } J_{\bbf}(\bbw)\in\mathbb{R}^{m\times d}, \bbu\in \mathbb{R}^m
\end{align*}
然后反向传播就很简单了：先计算损失的梯度，再按正确的顺序组合这些 **vjp** 函数。


### 例子：加偏置

我们从加偏置这个简单例子开始。


In [1]:
import numpy as np

In [2]:
def add(x, b):
    return x + b

def add_make_vjp(x, b):
    def vjp(u):
        return u, u
    return vjp

add.make_vjp = add_make_vjp

In [3]:
rng = np.random.RandomState(0)
x = rng.random((30,2)).astype('float32')
b_source  = np.array([1.])

In [4]:
xb = add(x,b_source)
np.allclose(xb, x+b_source)

True

In [5]:
vjp_add = add.make_vjp(x,b_source)

In [6]:
grad_x, grad_b = vjp_add(rng.random((30,2)))

In [7]:
grad_x.shape

(30, 2)

### 练习：点积与平方损失

实现对应的 vjp 函数。（注意：这里我们对平方损失稍微滥用了一点记号，因为目标 `y` 不是参数，不应该被更新！而且，平方损失的 vjp_{y_pred} 函数不依赖于它的输入 `u`）


In [8]:
def dot(x, W):
    return np.dot(x, W)

def dot_make_vjp(x, W):
    def vjp(u):
        return np.dot(u, W.T), np.einsum('na,nb-> nab',x , u)
    return vjp

dot.make_vjp = dot_make_vjp

def squared_loss(y_pred, y):
    return np.array([np.sum((y - y_pred) ** 2)])

def squared_loss_make_vjp(y_pred, y):
    def vjp(u):
        diff = y_pred - y
        return 2*diff, np.zeros_like(y)
    return vjp

squared_loss.make_vjp = squared_loss_make_vjp

## 准备工作

和 [02b_linear_reg.ipynb](https://github.com/dataflowr/notebooks/blob/master/Module2/02b_linear_reg.ipynb) 一样，我们的模型是：
$$
y_t = 2x^1_t-3x^2_t+1, \quad t\in\{1,\dots,30\}
$$

我们的任务是在给定'观测值' $(x_t,y_t)_{t\in\{1,\dots,30\}}$ 的情况下，恢复出权重 $w^1=2, w^2=-3$ 和偏置 $b = 1$。

为此，我们要解下面这个优化问题：
$$
\underset{w^1,w^2,b}{\operatorname{argmin}} \sum_{t=1}^{30} \left(w^1x^1_t+w^2x^2_t+b-y_t\right)^2
$$


In [9]:
rng = np.random.RandomState(0)
# 生成随机输入数据
x = rng.random((30,2)).astype('float32')
# 根据输入数据 x 生成对应的标签
y = np.dot(x, [2., -3.]) + 1.
y = np.expand_dims(y, axis=1).astype('float32')
w_source = np.array([2., -3.])
b_source  = np.array([1.])

In [10]:
def create_feed_forward(y, seed=0):
    rng = np.random.RandomState(seed)
    funcs = [dot,add,squared_loss]
    params = [rng.randn(2,1),rng.randn(1),y]
    return funcs, params

### 前向传播

下面的函数应该接受一批输入、函数以及它们的参数，并返回最终的值或全部的值。


In [11]:
def evaluate_chain(x, funcs, params, return_all=False):
    all_x = [x]
    for (f,p) in zip(funcs,params):
        x = f(all_x[-1],p)
        all_x.append(x)
    
    if return_all:
        return all_x
    else:
        return x

In [12]:
funcs, params = create_feed_forward(y=y, seed=0)
W, b, _ = params

In [13]:
xs = evaluate_chain(x, funcs, params, return_all=True)

### 反向传播

下面的函数应该先做前向传播，再做反向传播。


In [14]:
def backward_diff_chain(x, funcs, params):
    """
    Reverse-mode differentiation of a chain of computations.

    Args:
    x: initial input to the chain.
    funcs: a list of functions of the form func(x, param).
    params: a list of parameters, with len(params) = len(funcs).
    Returns:
    value, vjp_x, all vjp_params
    """
    # 对前馈模型求值，并保存中间计算结果，
    # 因为反向传播时会用到它们。
    xs = evaluate_chain(x, funcs, params, return_all=True)
    K = len(funcs)  # 函数个数。
    u = None # u = None # 损失的梯度不需要输入
    # 保存每个函数关于参数的雅可比矩阵的列表。
    J = [None] * K

    for (k,(f,p,x)) in reversed(list(enumerate(zip(funcs,params,xs)))):
        vjp_x, vjp_param = f.make_vjp(x,p)(u)
        u = vjp_x
        J[k] = vjp_param

    return xs[-1], u, J

In [15]:
loss, grad_x, grads = backward_diff_chain(x, funcs, params)

### 优化器

先计算每个参数的更新量，再修改参数。


In [16]:
def optim_SGD(grads, learning_rate = 1e-2):
    return [-learning_rate*g.sum(0) for i,g in enumerate(grads)]

def update_params(updates, params):
    return [params[i] + u for i,u in enumerate(updates)]

### 训练循环


In [17]:
funcs, params = create_feed_forward(y=y, seed=0)
W, b, _ = params
for epoch in range(10):
    loss, grad_x, grads = backward_diff_chain(x, funcs, params)
    print("progress:", "epoch:", epoch, "loss",loss)
    updates = optim_SGD(grads)
    params = update_params(updates, params)
    
# 训练结束后
print("estimation of the parameters:")
print(params[:-1])

progress: epoch: 0 loss [110.2900527]
progress: epoch: 1 loss [17.17577308]
progress: epoch: 2 loss [15.48445439]
progress: epoch: 3 loss [14.27434616]
progress: epoch: 4 loss [13.16495088]
progress: epoch: 5 loss [12.14630386]
progress: epoch: 6 loss [11.21068817]
progress: epoch: 7 loss [10.35106573]
progress: epoch: 8 loss [9.56101153]
progress: epoch: 9 loss [8.83465916]
estimation of the parameters:
[array([[ 1.6269806 ],
       [-1.09195793]]), array([0.121909])]


### Jax 实现

参见 [linear_regression_jax.ipynb](https://github.com/dataflowr/notebooks/blob/master/Module2/linear_regression_jax.ipynb)


In [18]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax
from functools import partial

class config:
    size_out = 1
    w_source = jnp.array(W)
    b_source = jnp.array(b)
    
def _linear(x, config):
    return hk.Linear(config.size_out,w_init=hk.initializers.Constant(config.w_source), b_init=hk.initializers.Constant(config.b_source))(x)

def mse_loss(y_pred, y_t):
    return jax.lax.integer_pow(y_pred - y_t,2).sum()

def loss_fn(x_in, y_t, config):
    return mse_loss(_linear(x=x_in, config=config),y_t)

hk_loss_fn = hk.without_apply_rng(hk.transform(partial(loss_fn, config=config)))
params = hk_loss_fn.init(x_in=x,y_t=y,rng=None)
loss_fn = hk_loss_fn.apply

optimizer = optax.sgd(learning_rate=1e-2)

opt_state = optimizer.init(params)
for epoch in range(10):
    loss, grads = jax.value_and_grad(loss_fn)(params,x_in=x,y_t=y)
    print("progress:", "epoch:", epoch, "loss",loss)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    
# 训练结束后
print("estimation of the parameters:")
print(params)

progress: epoch: 0 loss 110.29005
progress: epoch: 1 loss 17.175777
progress: epoch: 2 loss 15.484457
progress: epoch: 3 loss 14.274347
progress: epoch: 4 loss 13.164951
progress: epoch: 5 loss 12.146305
progress: epoch: 6 loss 11.21069
progress: epoch: 7 loss 10.351067
progress: epoch: 8 loss 9.561012
progress: epoch: 9 loss 8.83466
estimation of the parameters:
FlatMap({
  'linear': FlatMap({
              'b': DeviceArray([0.12190904], dtype=float32),
              'w': DeviceArray([[ 1.6269805],
                                [-1.0919579]], dtype=float32),
            }),
})


### PyTorch 实现

参见 [02b_linear_reg.ipynb](https://github.com/dataflowr/notebooks/blob/master/Module2/02b_linear_reg.ipynb)


In [19]:
import torch

dtype = torch.FloatTensor
w_init_t = torch.from_numpy(W).type(dtype)
b_init_t = torch.from_numpy(b).type(dtype)
x_t = torch.from_numpy(x).type(dtype)
y_t = torch.from_numpy(y).type(dtype)

model = torch.nn.Sequential(torch.nn.Linear(2, 1),)

for m in model.children():
    m.weight.data = w_init_t.T.clone()
    m.bias.data = b_init_t.clone()

loss_fn = torch.nn.MSELoss(reduction='sum')

model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

for epoch in range(10):
    y_pred = model(x_t)
    loss = loss_fn(y_pred, y_t)
    print("progress:", "epoch:", epoch, "loss",loss.item())
    # 梯度清零、反向传播、更新权重。
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
# 训练结束后
print("estimation of the parameters:")
for param in model.parameters():
    print(param)

progress: epoch: 0 loss 110.2900619506836
progress: epoch: 1 loss 17.1757755279541
progress: epoch: 2 loss 15.484455108642578
progress: epoch: 3 loss 14.274348258972168
progress: epoch: 4 loss 13.164952278137207
progress: epoch: 5 loss 12.146303176879883
progress: epoch: 6 loss 11.210689544677734
progress: epoch: 7 loss 10.351065635681152
progress: epoch: 8 loss 9.561013221740723
progress: epoch: 9 loss 8.834660530090332
estimation of the parameters:
Parameter containing:
tensor([[ 1.6270, -1.0920]], requires_grad=True)
Parameter containing:
tensor([0.1219], requires_grad=True)


[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)